## 1. Environment Setup and Library Imports

In [1]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

# -------------------------------------------------------------------------
# 1. Path resolution to access root directory and 'src' modules
# -------------------------------------------------------------------------
PROJECT_ROOT = Path.cwd().parent if "__file__" not in globals() else Path(__file__).resolve().parent.parent
DATA_DIR = PROJECT_ROOT / "data"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Import local custom modules
from src.pkldataset import PKLDataset
from src.helpers import set_seed, get_model, train_model

# -------------------------------------------------------------------------
# 2. Hardware and Training Hyperparameters (HC Thickness Prediction - Decision Fusion)
# -------------------------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
criterion = nn.CrossEntropyLoss()

target_material = "HC"
INPUT_CHANNELS = 1  # Each branch operates on 1 single-modality channel

# Classification setup: 5 target sheet thicknesses
thickness_list = ["T185", "T188", "T191", "T194", "T197"]
thickness_to_id = {t: idx for idx, t in enumerate(thickness_list)}
NUM_CLASSES_THICKNESS = len(thickness_list)  # 5 classes
NUM_EPOCHS = 35

# Sample sizes to evaluate sample efficiency (stratified across the 5 classes)
sample_sizes = [20, 50, 100, 200, 300, 400, 500]
subset_names = [f"train_{s}" for s in sample_sizes]

# Random seeds (10 seeds matching benchmark)
seeds = [101, 202, 303, 404, 505, 606, 707, 808, 909, 1001]

# -------------------------------------------------------------------------
# 3. Path Verification for Both Modalities (RP & AZ)
# -------------------------------------------------------------------------
print("==================================================================")
print("STEP 1: Verifying HC Paths for Decision Fusion (RP + AZ)...")
print("==================================================================")

all_paths_ok = True
hc_paths = {"RP": {}, "AZ": {}}

for t in thickness_list:
    for m in ["RP", "AZ"]:
        p = DATA_DIR / target_material / t / m
        exists = p.exists()
        if not exists:
            all_paths_ok = False
        print(f"[{target_material} {t} {m}] {p} -> {'FOUND' if exists else 'MISSING'}")
        hc_paths[m][t] = str(p)

print(f"\nPath status: {'ALL PATHS OK' if all_paths_ok else 'WARNING: SOME PATHS ARE MISSING'}")
print(f"Device: {device}")
print(f"Target Material: {target_material}")
print(f"Modalities: RP (Force) and AZ (Acceleration)")
print(f"Classes: {thickness_to_id} ({NUM_CLASSES_THICKNESS} classes)")
print(f"Subsets: {subset_names}")
print(f"Total Seeds: {len(seeds)}")

STEP 1: Verifying HC Paths for Decision Fusion (RP + AZ)...
[HC T185 RP] c:\Users\antho\OneDrive\Escritorio\Erstsemester_Anthony\Bachelorthesis\Training_Daten_Stanzen\Data\Data\data\HC\T185\RP -> FOUND
[HC T185 AZ] c:\Users\antho\OneDrive\Escritorio\Erstsemester_Anthony\Bachelorthesis\Training_Daten_Stanzen\Data\Data\data\HC\T185\AZ -> FOUND
[HC T188 RP] c:\Users\antho\OneDrive\Escritorio\Erstsemester_Anthony\Bachelorthesis\Training_Daten_Stanzen\Data\Data\data\HC\T188\RP -> FOUND
[HC T188 AZ] c:\Users\antho\OneDrive\Escritorio\Erstsemester_Anthony\Bachelorthesis\Training_Daten_Stanzen\Data\Data\data\HC\T188\AZ -> FOUND
[HC T191 RP] c:\Users\antho\OneDrive\Escritorio\Erstsemester_Anthony\Bachelorthesis\Training_Daten_Stanzen\Data\Data\data\HC\T191\RP -> FOUND
[HC T191 AZ] c:\Users\antho\OneDrive\Escritorio\Erstsemester_Anthony\Bachelorthesis\Training_Daten_Stanzen\Data\Data\data\HC\T191\AZ -> FOUND
[HC T194 RP] c:\Users\antho\OneDrive\Escritorio\Erstsemester_Anthony\Bachelorthesis\Trai

## 2. Data Ingestion & In-Memory Preloading (Fusion)

In [2]:
import os
import torch
from torch.utils.data import DataLoader
from src.pkldataset import PKLDataset

# -------------------------------------------------------------------------
# STEP 2: Load and Synchronize RP and AZ Signals for All 5 Thicknesses
# -------------------------------------------------------------------------
print("==================================================================")
print("STEP 2: Preloading and Synchronizing RP & AZ Data into RAM...")
print("==================================================================")

cpu_cores = os.cpu_count() or 4
num_workers = min(8, cpu_cores)

# Container: {'inputs_rp': tensor, 'inputs_az': tensor, 'targets': tensor}
hc_sync_tensors = {}

for t_name in thickness_list:
    t_id = thickness_to_id[t_name]
    
    # 1. Load RP signals
    ds_rp = PKLDataset(folder_path=hc_paths["RP"][t_name], modality="single")
    loader_rp = DataLoader(ds_rp, batch_size=512, shuffle=False, num_workers=num_workers)
    xs_rp = [x for x, _ in loader_rp]
    inputs_rp = torch.cat(xs_rp, dim=0)
    
    # 2. Load AZ signals
    ds_az = PKLDataset(folder_path=hc_paths["AZ"][t_name], modality="single")
    loader_az = DataLoader(ds_az, batch_size=512, shuffle=False, num_workers=num_workers)
    xs_az = [x for x, _ in loader_az]
    inputs_az = torch.cat(xs_az, dim=0)
    
    # Verify temporal alignment
    min_len = min(inputs_rp.shape[0], inputs_az.shape[0])
    inputs_rp = inputs_rp[:min_len]
    inputs_az = inputs_az[:min_len]
    targets = torch.full((min_len,), fill_value=t_id, dtype=torch.long)
    
    hc_sync_tensors[t_name] = {
        "inputs_rp": inputs_rp,
        "inputs_az": inputs_az,
        "targets": targets
    }
    
    print(f"[{t_name}] Synchronized: {min_len} samples | RP: {tuple(inputs_rp.shape)} | AZ: {tuple(inputs_az.shape)}")

print("\nAll RP and AZ tensors successfully synchronized and cached in RAM.")

STEP 2: Preloading and Synchronizing RP & AZ Data into RAM...
[T185] Synchronized: 5010 samples | RP: (5010, 1, 2800) | AZ: (5010, 1, 2800)
[T188] Synchronized: 5010 samples | RP: (5010, 1, 2800) | AZ: (5010, 1, 2800)
[T191] Synchronized: 5010 samples | RP: (5010, 1, 2800) | AZ: (5010, 1, 2800)
[T194] Synchronized: 5009 samples | RP: (5009, 1, 2800) | AZ: (5009, 1, 2800)
[T197] Synchronized: 5010 samples | RP: (5010, 1, 2800) | AZ: (5010, 1, 2800)

All RP and AZ tensors successfully synchronized and cached in RAM.


## 3. Train/Test Partitioning - Fusion

In [3]:
import numpy as np
import torch
from torch.utils.data import TensorDataset

# -------------------------------------------------------------------------
# STEP 3: Train/Test Partitioning for Synchronized Multi-Modal Evaluation
# -------------------------------------------------------------------------
print("==================================================================")
print("STEP 3: Partitioning Train Pools and Dual Test Sets...")
print("==================================================================")

TEST_RATIO = 0.20
EVAL_SEED = 42

np.random.seed(EVAL_SEED)

train_pools_sync = {}
test_rp_list, test_az_list, test_targets_list = [], [], []

for t_name in thickness_list:
    rp_data = hc_sync_tensors[t_name]["inputs_rp"]
    az_data = hc_sync_tensors[t_name]["inputs_az"]
    targets = hc_sync_tensors[t_name]["targets"]
    n_samples = rp_data.shape[0]
    
    indices = np.random.permutation(n_samples)
    n_test = int(n_samples * TEST_RATIO)
    
    test_idx = indices[:n_test]
    train_idx = indices[n_test:]
    
    train_pools_sync[t_name] = {
        "inputs_rp": rp_data[train_idx],
        "inputs_az": az_data[train_idx],
        "targets": targets[train_idx]
    }
    
    test_rp_list.append(rp_data[test_idx])
    test_az_list.append(az_data[test_idx])
    test_targets_list.append(targets[test_idx])
    
    print(f"[{t_name}] Train pool: {len(train_idx)} | Test set: {len(test_idx)}")

# Consolidated Test Sets on GPU
X_test_rp = torch.cat(test_rp_list, dim=0).to(device)
X_test_az = torch.cat(test_az_list, dim=0).to(device)
Y_test_fusion = torch.cat(test_targets_list, dim=0).to(device)

print(f"\nConsolidated Test RP: {tuple(X_test_rp.shape)}")
print(f"Consolidated Test AZ: {tuple(X_test_az.shape)}")
print(f"Consolidated Test Targets: {tuple(Y_test_fusion.shape)}")

# -------------------------------------------------------------------------
# Stratified Multi-Modal Sampler
# -------------------------------------------------------------------------
def get_balanced_sync_subset(train_pools, total_samples, seed=42):
    np.random.seed(seed)
    num_classes = len(thickness_list)
    samples_per_class = total_samples // num_classes
    
    sel_rp, sel_az, sel_y = [], [], []
    
    for t_name in thickness_list:
        pool_rp = train_pools[t_name]["inputs_rp"]
        pool_az = train_pools[t_name]["inputs_az"]
        pool_y = train_pools[t_name]["targets"]
        total_avail = pool_rp.shape[0]
        
        if total_avail < samples_per_class:
            raise ValueError(f"Not enough samples in {t_name}: {total_avail} < {samples_per_class}")
            
        chosen = np.random.choice(total_avail, size=samples_per_class, replace=False)
        sel_rp.append(pool_rp[chosen])
        sel_az.append(pool_az[chosen])
        sel_y.append(pool_y[chosen])
        
    b_rp = torch.cat(sel_rp, dim=0)
    b_az = torch.cat(sel_az, dim=0)
    b_y = torch.cat(sel_y, dim=0)
    
    perm = torch.randperm(b_rp.size(0))
    return b_rp[perm], b_az[perm], b_y[perm]

# Container tracking all three streams: RP, AZ, and Decision Fusion
results = {
    s_name: {
        "overall_rp": [],
        "overall_az": [],
        "overall_fusion": [],
        **{f"fusion_{t}": [] for t in thickness_list}
    }
    for s_name in subset_names
}

print(f"\nStep 3 completed. Results container configured for Decision Fusion.")

STEP 3: Partitioning Train Pools and Dual Test Sets...
[T185] Train pool: 4008 | Test set: 1002
[T188] Train pool: 4008 | Test set: 1002
[T191] Train pool: 4008 | Test set: 1002
[T194] Train pool: 4008 | Test set: 1001
[T197] Train pool: 4008 | Test set: 1002

Consolidated Test RP: (5009, 1, 2800)
Consolidated Test AZ: (5009, 1, 2800)
Consolidated Test Targets: (5009,)

Step 3 completed. Results container configured for Decision Fusion.


## 4. Training and Evaluation - Fusion

In [4]:
import torch
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

# -------------------------------------------------------------------------
# 1. Fast Dual In-Memory Evaluation Helper (Soft Voting Fusion)
# -------------------------------------------------------------------------
def fast_eval_decision_fusion(model_rp, model_az, x_rp, x_az, y_test, batch_size=512):
    model_rp.eval()
    model_az.eval()
    
    all_preds_rp, all_preds_az, all_preds_fusion = [], [], []
    total = y_test.size(0)
    
    with torch.no_grad():
        for i in range(0, total, batch_size):
            bx_rp = x_rp[i:i + batch_size]
            bx_az = x_az[i:i + batch_size]
            
            # Forward passes
            logits_rp = model_rp(bx_rp)
            logits_az = model_az(bx_az)
            
            # Softmax class probability distributions
            prob_rp = F.softmax(logits_rp, dim=1)
            prob_az = F.softmax(logits_az, dim=1)
            
            # Equal-weighted late fusion (Average Probability)
            prob_fusion = 0.5 * (prob_rp + prob_az)
            
            all_preds_rp.append(prob_rp.argmax(dim=1))
            all_preds_az.append(prob_az.argmax(dim=1))
            all_preds_fusion.append(prob_fusion.argmax(dim=1))
            
    preds_rp = torch.cat(all_preds_rp, dim=0)
    preds_az = torch.cat(all_preds_az, dim=0)
    preds_fusion = torch.cat(all_preds_fusion, dim=0)
    
    acc_rp = 100.0 * (preds_rp == y_test).sum().item() / total
    acc_az = 100.0 * (preds_az == y_test).sum().item() / total
    acc_fusion = 100.0 * (preds_fusion == y_test).sum().item() / total
    
    # Class-specific accuracies for Decision Fusion
    class_accs_fusion = {}
    for t_name, t_id in thickness_to_id.items():
        mask = (y_test == t_id)
        if mask.sum().item() > 0:
            c_acc = 100.0 * (preds_fusion[mask] == t_id).sum().item() / mask.sum().item()
            class_accs_fusion[t_name] = c_acc
        else:
            class_accs_fusion[t_name] = 0.0
            
    return acc_rp, acc_az, acc_fusion, class_accs_fusion

# -------------------------------------------------------------------------
# 2. Execution Loop across Seeds and Subsets
# -------------------------------------------------------------------------
print("==================================================================")
print("STEP 4: Training Dual 1D-CNNs and Evaluating Decision Fusion...")
print("==================================================================")

for seed in seeds:
    print(f"\n==========================================")
    print(f">>> Running Decision Fusion for Seed: {seed}")
    print(f"==========================================")
    set_seed(seed)
    
    for s_size, s_name in zip(sample_sizes, subset_names):
        # 1. Synchronized stratified extraction
        train_rp, train_az, train_y = get_balanced_sync_subset(
            train_pools=train_pools_sync,
            total_samples=s_size,
            seed=seed
        )
        
        # Adaptive batch sizing
        if s_size < 64:
            curr_batch_size = max(8, s_size // NUM_CLASSES_THICKNESS)
        elif s_size < 200:
            curr_batch_size = 32
        else:
            curr_batch_size = 64
            
        loader_rp = DataLoader(TensorDataset(train_rp, train_y), batch_size=curr_batch_size, shuffle=True)
        loader_az = DataLoader(TensorDataset(train_az, train_y), batch_size=curr_batch_size, shuffle=True)
        
        # 2. Train Branch 1: RP (Force)
        model_rp = get_model(input_channels=1, num_classes=NUM_CLASSES_THICKNESS).to(device)
        opt_rp = optim.Adam(model_rp.parameters(), lr=7e-4, weight_decay=1e-4)
        sched_rp = optim.lr_scheduler.StepLR(opt_rp, step_size=20, gamma=0.2)
        model_rp = train_model(model_rp, loader_rp, criterion, opt_rp, sched_rp, NUM_EPOCHS, device)
        
        # 3. Train Branch 2: AZ (Acceleration)
        model_az = get_model(input_channels=1, num_classes=NUM_CLASSES_THICKNESS).to(device)
        opt_az = optim.Adam(model_az.parameters(), lr=7e-4, weight_decay=1e-4)
        sched_az = optim.lr_scheduler.StepLR(opt_az, step_size=20, gamma=0.2)
        model_az = train_model(model_az, loader_az, criterion, opt_az, sched_az, NUM_EPOCHS, device)
        
        # 4. Perform Decision Fusion on Consolidated Test Set
        acc_rp, acc_az, acc_fusion, class_accs = fast_eval_decision_fusion(
            model_rp=model_rp,
            model_az=model_az,
            x_rp=X_test_rp,
            x_az=X_test_az,
            y_test=Y_test_fusion,
            batch_size=512
        )
        
        # Store metrics
        results[s_name]["overall_rp"].append(acc_rp)
        results[s_name]["overall_az"].append(acc_az)
        results[s_name]["overall_fusion"].append(acc_fusion)
        for t_name in thickness_list:
            results[s_name][f"fusion_{t_name}"].append(class_accs[t_name])
            
        print(f"[Seed {seed}] {s_name} ({s_size:3d}) | RP: {acc_rp:5.2f}% | AZ: {acc_az:5.2f}% | FUSION: {acc_fusion:5.2f}%")

print("\nStep 4 completed successfully.")

STEP 4: Training Dual 1D-CNNs and Evaluating Decision Fusion...

>>> Running Decision Fusion for Seed: 101
Epoch 1/35, Loss: 2.1767
Epoch 2/35, Loss: 2.7331
Epoch 3/35, Loss: 1.9435
Epoch 4/35, Loss: 1.6956
Epoch 5/35, Loss: 1.9445
Epoch 6/35, Loss: 1.5836
Epoch 7/35, Loss: 1.4521
Epoch 8/35, Loss: 1.1823
Epoch 9/35, Loss: 1.4279
Epoch 10/35, Loss: 1.1708
Epoch 11/35, Loss: 1.1592
Epoch 12/35, Loss: 0.8178
Epoch 13/35, Loss: 0.9157
Epoch 14/35, Loss: 0.9839
Epoch 15/35, Loss: 0.8576
Epoch 16/35, Loss: 0.8767
Epoch 17/35, Loss: 1.0159
Epoch 18/35, Loss: 0.6370
Epoch 19/35, Loss: 0.6718
Epoch 20/35, Loss: 0.6410
Epoch 21/35, Loss: 0.5934
Epoch 22/35, Loss: 0.4843
Epoch 23/35, Loss: 0.7169
Epoch 24/35, Loss: 0.6596
Epoch 25/35, Loss: 0.5877
Epoch 26/35, Loss: 0.6004
Epoch 27/35, Loss: 0.4742
Epoch 28/35, Loss: 0.4325
Epoch 29/35, Loss: 0.6434
Epoch 30/35, Loss: 0.3954
Epoch 31/35, Loss: 0.4069
Epoch 32/35, Loss: 0.5647
Epoch 33/35, Loss: 0.5247
Epoch 34/35, Loss: 0.4496
Epoch 35/35, Loss:

## 5. Summary of Results - Fusion

In [7]:
import numpy as np
import pandas as pd
from pathlib import Path

# -------------------------------------------------------------------------
# STEP 5: Compile Final Summary Table and Export (HC Thickness - Decision Fusion)
# -------------------------------------------------------------------------
print("==================================================================")
print("STEP 5: Compiling Consolidated Results Table (HC Thickness - Decision Fusion)...")
print("==================================================================")

summary_data = []

for s_name in subset_names:
    row = {"Subset": s_name}
    
    # Per-class accuracy (recall) for each sheet thickness
    for t_name in thickness_list:
        # Detecta automáticamente si se guardó como 'T185' o como 'fusion_T185'
        if t_name in results[s_name]:
            class_scores = results[s_name][t_name]
        elif f"fusion_{t_name}" in results[s_name]:
            class_scores = results[s_name][f"fusion_{t_name}"]
        else:
            raise KeyError(f"No se encontró ni '{t_name}' ni 'fusion_{t_name}' en results['{s_name}']. Claves presentes: {list(results[s_name].keys())}")
            
        mean_c = np.mean(class_scores)
        std_c = np.std(class_scores)
        row[t_name] = f"{mean_c:.2f} ± {std_c:.2f}"
        
    summary_data.append(row)

# Strict column ordering: Subset -> T185 ... T197
column_order = ["Subset"] + thickness_list
df_thickness_fusion = pd.DataFrame(summary_data)[column_order]

print(f"\nConsolidated Thickness Prediction Results (HC - Decision Fusion) [Mean ± Std over {len(seeds)} seeds]:")
display(df_thickness_fusion)

# Save to CSV
output_csv = "thickness_pred_HC_decision_fusion_summary.csv"
df_thickness_fusion.to_csv(output_csv, index=False)
print(f"\nFinal results successfully exported to: {output_csv}")

STEP 5: Compiling Consolidated Results Table (HC Thickness - Decision Fusion)...

Consolidated Thickness Prediction Results (HC - Decision Fusion) [Mean ± Std over 10 seeds]:


,Subset,T185,T188,T191,T194,T197
0,train_20,64.75 ± 11.93,53.58 ± 11.78,56.34 ± 13.16,64.37 ± 15.87,76.45 ± 17.46
1,train_50,87.79 ± 8.14,82.70 ± 11.85,82.91 ± 7.01,87.02 ± 7.88,95.72 ± 9.01
2,train_100,92.48 ± 4.53,88.11 ± 3.46,89.01 ± 3.22,93.76 ± 3.07,99.59 ± 0.39
3,train_200,94.67 ± 2.17,92.91 ± 1.83,92.46 ± 1.99,96.09 ± 1.76,99.61 ± 0.44
4,train_300,95.42 ± 1.88,94.90 ± 1.36,95.00 ± 1.78,97.05 ± 1.68,99.65 ± 0.32
5,train_400,96.35 ± 1.82,94.60 ± 1.51,95.21 ± 1.51,97.86 ± 1.27,99.84 ± 0.14
6,train_500,96.95 ± 0.91,95.97 ± 1.00,95.96 ± 1.41,98.33 ± 0.86,99.71 ± 0.27



Final results successfully exported to: thickness_pred_HC_decision_fusion_summary.csv
